In [1]:
# @title Setup, Installation, and api key management

# Install ADK and LiteLLM
!pip install google-adk -q
!pip install litellm -q

print("Installation complete.")

Installation complete.


In [2]:
# @title Imports, Settings and Constants
from google.adk.agents import Agent, LlmAgent, LoopAgent, SequentialAgent
from google.adk.models import LlmResponse, LlmRequest
from google.adk.agents.callback_context import CallbackContext
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types # For creating message Content/Parts
from google.adk.tools import google_maps_grounding
from google.adk.tools.tool_context import ToolContext
from typing import Optional, Dict, Any
from google.adk.tools.base_tool import BaseTool
#from google.adk.tools import VertexAiSearchTool
from google.adk.tools import google_search, agent_tool, tool_context

PROJECT_ID = "qwiklabs-gcp-04-50597814ce11"
LOCATION = "us-east4"
STAGING_BUCKET = "qwiklabs-gcp-04-50597814ce11-reasoning-engine-staging"
DEFAULT_AGENT_MODEL = "gemini-2.5-flash"
USER_ID = "agent-engine-test-user"
# More supported models can be referenced here: https://ai.google.dev/gemini-api/docs/models#model-variations
MODEL_GEMINI_2_5_FLASH = "gemini-2.5-flash"

print("\nEnvironment configured.")



Environment configured.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [3]:
# @title Vertex AI & GCS Setup
import vertexai
from google.cloud import storage

# Ensure the bucket exists
storage_client = storage.Client(project=PROJECT_ID)
try:
    bucket = storage_client.get_bucket(STAGING_BUCKET)
    print(f"Bucket {STAGING_BUCKET} already exists.")
except Exception:
    print(f"Creating bucket {STAGING_BUCKET}...")
    storage_client.create_bucket(STAGING_BUCKET, location=LOCATION)

vertexai.init(
   project=PROJECT_ID,
   location=LOCATION,
   staging_bucket=f"gs://{STAGING_BUCKET}",
)

Bucket qwiklabs-gcp-04-50597814ce11-reasoning-engine-staging already exists.


In [4]:
# @title log_user_prompt

def log_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Logs the user prompts."""

    agent_name = callback_context.agent_name
    log_header = f"[Callback:{agent_name}:log_user_prompt]"

    # Inspect the last user message in the request contents
    last_user_message = ""
    if llm_request.contents and llm_request.contents[-1].role == 'user':
         if llm_request.contents[-1].parts:
            last_user_message = llm_request.contents[-1].parts[0].text

    if last_user_message == None or last_user_message == "":
        return None
    print(f"{log_header} Logging last user message: '{last_user_message}'")
    return None


In [23]:
# @title log_model_response

def log_model_response(
    callback_context: CallbackContext, llm_response: LlmResponse
) -> Optional[LlmResponse]:
    """Logs the model's responses."""

    agent_name = callback_context.agent_name
    log_header = f"[Callback:{agent_name}:log_model_response]"

    # --- Inspection ---
    original_text = ""
    if llm_response.content and llm_response.content.parts:
        # Assuming simple text response for this example
        if llm_response.content.parts[0].text:
            original_text = llm_response.content.parts[0].text
            print(f"{log_header} Inspected original response text: '{original_text[:100]}...'") # Log snippet
        elif llm_response.content.parts[0].function_call:
             print(f"{log_header} Inspected response: Contains function call '{llm_response.content.parts[0].function_call.name}'. No text modification.")
             return None # Don't modify tool calls in this example
        else:
             print(f"{log_header} Inspected response: No text content found.")
             return None
    elif llm_response.error_message:
        print(f"{log_header} Inspected response: Contains error '{llm_response.error_message}'. No modification.")
        return None
    else:
        print(f"{log_header} Inspected response: Empty LlmResponse.")
        return None # Nothing to modify

    return None

In [6]:
# @title State management

# def append_to_state(tool_context, field, response):
#    existing_state = tool_context.state.get(field, [])
#    tool_context.state[field] = existing_state + [response]
#    return {"status": "success"}

def read_suggestion(tool_context):
  """
  USE THIS TO READ THE A NEW SUGGESTED TOPIC
  This should just be a single line statement.
  """
  content = tool_context.state.get("cheatsheet_topic")
  if content is None:
    return "There are no cheatsheet topics."
  return content

def add_suggestion(tool_context, value):
  """
  USE THIS TO ADD A NEW SUGGESTED TOPIC
  This should just be a single line statement.
  """
  tool_context.state["cheatsheet_topic"] = value
  return {"status": "success", "message": "Cheatsheet topic updated."}

def read_cheatsheet(tool_context):
  """
  USE THIS TO READ THE ENTIRE DOCUMENT.
  This cheatsheet is in Markdown format.
  """
  content = tool_context.state.get("cheatsheet_document")
  if content is None:
    return "The cheatsheet is currently empty."
  return content

def replace_cheatsheet(tool_context, value):
  """
  USE THIS TO OVERWRITE THE ENTIRE DOCUMENT.
  This deletes the old cheatsheet and replaces it with the new 'value'.
  Use this for new versions or entirely new topics.
  """
  if(value.strip() == ""):
    return {"status": "error", "message": "Cheatsheet cannot be empty."}
  tool_context.state["cheatsheet_document"] = value
  #print(tool_context.state["cheatsheet_document"])
  return {"status": "success", "message": "Cheatsheet updated."}

def print_cheatsheet(tool_context):
  """
  USE THIS TO PRINT THE ENTIRE DOCUMENT.
  This cheatsheet is in Markdown format.
  """
  content = tool_context.state.get("cheatsheet_document", "Empty")
  print(f"{content}")
  return "Cheatsheet printed."

In [7]:
# @title print_cheatsheet_callback

# def print_cheatsheet_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
#     """Prints the current cheatsheet."""

def print_cheatsheet_callback(tool: BaseTool, args: Dict[str, Any], tool_context: ToolContext, tool_response: Dict) -> Optional[Dict]:
  """Prints the current cheatsheet."""

  agent_name = tool_context.agent_name
  log_header = f"[Callback:{agent_name}:print_cheatsheet]"

  print(f"{log_header} Current cheatsheet document")
  print(tool_context.state["cheatsheet_document"])
  return None

In [8]:
# @title Define the Search Agent

AGENT_MODEL = MODEL_GEMINI_2_5_FLASH # Starting with Gemini

instruction = """
              You are a helpful search assistant.
              """

search_agent = Agent(
    name="search_agent",
    model=AGENT_MODEL, # Can be a string for Gemini or a LiteLlm object
    description="Provides generic search capabilities.",
    instruction=instruction,
    tools=[google_search],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response
)

print(f"Agent '{search_agent.name}' created using model '{AGENT_MODEL}'.")

Agent 'search_agent' created using model 'gemini-2.5-flash'.


In [9]:
# @title Define the Research Agent

AGENT_MODEL = MODEL_GEMINI_2_5_FLASH # Starting with Gemini

instruction = """
              You are a helpful technical research assistant.
              Find a new topic related to the research material.
              Identify a new topic that does not exist in the cheatsheet. You
              can see existing topics by using the 'read_cheatsheet' tool.
              You can suggest a new topic by using 'add_suggestion'.
              Only suggest new topics.
              Do not repeat topics.
              The cheatsheet is in markdown format.
              """

research_agent = Agent(
    name="research_agent",
    model=AGENT_MODEL, # Can be a string for Gemini or a LiteLlm object
    description="Provides a topic based on the research topic.",
    instruction=instruction,
    tools=[
        agent_tool.AgentTool(agent=search_agent),
        add_suggestion,
        read_cheatsheet
        ],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response
)

print(f"Agent '{research_agent.name}' created using model '{AGENT_MODEL}'.")

Agent 'research_agent' created using model 'gemini-2.5-flash'.


In [10]:
# @title Define the Technical Writer

AGENT_MODEL = MODEL_GEMINI_2_5_FLASH # Starting with Gemini

instruction = """
              You are a helpful technical writer.
              You build terse, logical, cheatsheets for technical users.
              You provide syntax and short descriptions of the suggested topic
              found using the 'read_suggestion' tool.
              You give technical details concerning the suggestion.
              Read the entire cheatsheet by using the 'read_cheatsheet' tool and
              create a new section with the suggestion. You then replace the
              previous cheatsheet using the 'replace_cheatsheet' tool with your
              updates using the 'replace_cheatsheet' tool.
              The cheatsheet is in markdown format.
              """

technical_writer = Agent(
    name="technical_writer",
    model=AGENT_MODEL, # Can be a string for Gemini or a LiteLlm object
    description="Writes technical documents.",
    instruction=instruction,
    tools=[
        agent_tool.AgentTool(agent=search_agent),
        read_cheatsheet,
        read_suggestion,
        replace_cheatsheet],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response
)

print(f"Agent '{research_agent.name}' created using model '{AGENT_MODEL}'.")

Agent 'research_agent' created using model 'gemini-2.5-flash'.


In [11]:
# @title Define the Critique Agent
AGENT_MODEL = MODEL_GEMINI_2_5_FLASH # Starting with Gemini

instruction = """
              You are a helpful critical assistant.
              Read the current cheatsheet by using the 'read_cheatsheet' tool
              and make sure it is terse informative and accurate. If it is okay,
              do nothing and say that you have nothing to do.
              If it is not okay, replace the previous cheatsheet with all your
              updates. Use the 'replace_cheatsheet' tool.
              DO NOT write an empty cheatsheet.
              It is okay if there is nothing to be done. Just say that you have nothing to do.
              The cheatsheet is in markdown format.
              """

critique_agent = Agent(
    name="critique_agent",
    model=AGENT_MODEL, # Can be a string for Gemini or a LiteLlm object
    description="Give a short, quick,  analysis of the material",
    instruction=instruction,
    tools=[
        read_cheatsheet,
        replace_cheatsheet
    ],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response
)

print(f"Agent '{critique_agent.name}' created using model '{AGENT_MODEL}'.")


Agent 'critique_agent' created using model 'gemini-2.5-flash'.


In [12]:
# @title Define the Technical Writers
AGENT_MODEL = MODEL_GEMINI_2_5_FLASH # Starting with Gemini

technical_writers = SequentialAgent(
    name="technical_writers",
    description="Find a new feature to write it in a cheatsheet.",
    sub_agents=[
        research_agent,
        technical_writer,
        critique_agent],
)

print(f"Agent '{technical_writers.name}' created using model '{AGENT_MODEL}'.")

Agent 'technical_writers' created using model 'gemini-2.5-flash'.


In [13]:
# @title Define the Writers Room Agent

AGENT_MODEL = MODEL_GEMINI_2_5_FLASH # Starting with Gemini

writers_room = LoopAgent(
    name="writers_room",
    description="Iterates over 3 new features based on the requrested cheatsheet technology",
    sub_agents=[technical_writers],
    max_iterations=3

)

print(f"Agent '{writers_room.name}' created using model '{AGENT_MODEL}'.")

Agent 'writers_room' created using model 'gemini-2.5-flash'.


In [14]:
# @title Define the Root Agent

AGENT_MODEL = MODEL_GEMINI_2_5_FLASH # Starting with Gemini

instruction = """
              You delegate the create of cheatsheets.
              If the user does not ask you to write a cheatsheet, then tell them that you,
              "only write technical cheatsheets".
              Start a new cheatsheet with a title based on the user's request.
              Use the 'replace_cheatsheet' tool to create a new cheatsheet.
              Call the 'writers_room' agent to write a cheatsheet.
              Once everything is done, call `print_cheatsheet` to print the cheatsheet.
              """

root_agent = LlmAgent(
    name="root_agent",
    model=AGENT_MODEL,
    description="Root agent for the cheatsheet writing app.",
    instruction=instruction,
    tools=[replace_cheatsheet, print_cheatsheet],
    sub_agents=[writers_room],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
    #after_tool_callback=print_cheatsheet_callback
)

print(f"Agent '{root_agent.name}' created using model '{AGENT_MODEL}'.")

Agent 'root_agent' created using model 'gemini-2.5-flash'.


In [15]:
# @title Use vertexai with root_agent

from vertexai.preview import reasoning_engines

app = reasoning_engines.AdkApp(
    agent=root_agent
)

In [16]:
# @title Create session
user_id = "test-user-id"
session = app.create_session(user_id=user_id)
session_id = session["id"]

#print(session_id)
#print(session)
print(f"Session created: {session_id}")

Session created: c04f8e0f-7cf1-4b38-ad33-df072bef5ad0


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:872: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


In [17]:
# @title callTheRootAgent (Modified)
from IPython.display import Markdown, display

def callTheRootAgent(prompt:str):
  response = ""
  cheatsheet_content = "No cheatsheet found."

  for event in app.stream_query(
      user_id=user_id,
      session_id=session["id"],
      message=prompt,
  ):
    # 1. Accumulate the text response chunks
    content = event.get("content", {})
    parts = content.get("parts", [])
    if parts and "text" in parts[0]:
      response += parts[0]["text"] # Append chunks if they are incremental

    # 2. Only grab the state if this specific event contains it
    # Most frameworks only send the full state in a 'metadata' or 'final' event
    state = event.get("state") or event.get("context")
    if state:
        new_val = state.get("cheatsheet_document")
        # Ensure we only update if the new value is actually populated
        if new_val:
            cheatsheet_content = new_val

  return {
      "response": response,
      "cheatsheet": cheatsheet_content
  }



# Example Usage:
# result = callTheRootAgent("Update the cheatsheet with new data")
# print(result["response"])
# print(result["cheatsheet"])


In [18]:
# @title Test - Create a .NET 11 cheatsheet

#prompt = "What is the weather in England?"
prompt = "Write a cheat sheet for the new features in .NET 11."

print(f"User prompt: {prompt}")

agent_response = callTheRootAgent(prompt)



User prompt: Write a cheat sheet for the new features in .NET 11.
[Callback:root_agent:log_user_prompt] Logging last user message: 'Write a cheat sheet for the new features in .NET 11.'


[Callback:root_agent:log_model_response] Inspected response: Contains function call 'replace_cheatsheet'. No text modification.
[Callback:research_agent:log_user_prompt] Logging last user message: 'For context:'
[Callback:research_agent:log_model_response] Inspected response: Contains function call 'read_cheatsheet'. No text modification.
[Callback:research_agent:log_model_response] Inspected response: Contains function call 'search_agent'. No text modification.
[Callback:search_agent:log_user_prompt] Logging last user message: 'new features in .NET 11'
[Callback:search_agent:log_model_response] Inspected original response text: '.NET 11, currently in its preview stages, introduces a wide array of new features and improvements f...'
[Callback:research_agent:log_model_response] Inspected response: Contains function call 'add_suggestion'. No text modification.
[Callback:research_agent:log_model_response] Inspected original response text: 'I have added "Runtime-native async (Runtime Async

In [19]:
# @title Display response
print(agent_response["response"])

I have added "Runtime-native async (Runtime Async V2)" as a new topic to the cheatsheet.The cheatsheet has been updated with the "Runtime-native async (Runtime Async V2)" section.I have updated the cheatsheet with a more comprehensive and organized list of new features in .NET 11, including details on Runtime and Performance, C# Language, ASP.NET Core, SDK/CLI, Libraries, and EF Core.I have added "CoreCLR on WebAssembly (foundational)" as a new topic suggestion.The cheatsheet has been updated with "CoreCLR on WebAssembly (foundational)" under the "Runtime and Performance" section.I have nothing to do. The cheatsheet is terse, informative, and accurate.I have added "GC heap hard-limit support (for 32-bit processes)" as a new topic.The cheatsheet has been updated with "GC heap hard-limit support (for 32-bit processes)" under the "Runtime and Performance" section.I have nothing to do. The cheatsheet is terse, informative, and accurate.


In [20]:
# @title Display cheatsheet

print(agent_response["cheatsheet"])

No cheatsheet found.


In [21]:
# @title Deploy to Agent Platform

from vertexai import agent_engines

# The staging_bucket is handled by the global vertexai.init configuration
remote_agent = agent_engines.create(
   app,
   requirements=["google-cloud-aiplatform[agent_engines,adk]"],
)


INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.12.3', 'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.157.0'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-04-50597814ce11-reasoning-engine-staging
INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-04-50597814ce11-reasoning-engine-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-50597814ce11-reasoning-engine-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-50597814ce11-reasoning-engine-staging/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creati

In [24]:
# @title Test Remote Agent

print(f"Targeting remote agent: {remote_agent.resource_name}")

# List available methods to verify the correct one
methods = [method for method in dir(remote_agent) if not method.startswith('_')]
print(f"Available methods: {methods}")

MESSAGE = "Write a cheat sheet for the new features in .NET 11."

try:
    print("\n--- Attempting Query ---")
    # In some SDK versions, the method is 'query' or 'predict'.
    # For ADK remote engines, 'stream_query' is the standard.
    response_iterator = remote_agent.stream_query(
       user_id=USER_ID,
       message=MESSAGE,
    )

    for i, event in enumerate(response_iterator):
        # Extract text safely from the event
        content = event.get("content", {})
        parts = content.get("parts", [])
        for part in parts:
            if isinstance(part, dict) and "text" in part:
                print(part["text"], end="")
            elif isinstance(part, str):
                print(part, end="")

except Exception as e:
    print(f"\nError: {e}")

Targeting remote agent: projects/172082414361/locations/us-east4/reasoningEngines/8252503269528567808
Available methods: ['api_client', 'async_add_session_to_memory', 'async_create_session', 'async_delete_session', 'async_get_session', 'async_list_sessions', 'async_search_memory', 'async_stream_query', 'client_class', 'create', 'create_session', 'create_time', 'credentials', 'delete', 'delete_session', 'display_name', 'encryption_spec', 'execution_api_client', 'execution_async_client', 'gca_resource', 'get_session', 'labels', 'list', 'list_sessions', 'location', 'name', 'operation_schemas', 'project', 'resource_name', 'stream_query', 'streaming_agent_run_with_events', 'to_dict', 'update', 'update_time', 'wait']

--- Attempting Query ---
